In [1]:
import numpy as np
import json
import pickle

from sklearn.datasets import make_multilabel_classification

from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier as BasePredictor

import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [2]:
import sys

sys.path.append("..")

In [3]:
from src.data.formatting import resample_data_set
from src.conformal_risk_control.fnr import compute_prediction_set
from src.model_selection.fnr_oracle import (
    compute_oracle_prediction_set,
    compute_oracle_orders_and_sizes,
)
from src.model_selection.fnr_upper import (
    compute_upper_prediction_set,
    compute_lower_orders_and_sizes,
    compute_upper_orders_and_sizes,
)
from src.model_selection.fnr_split import select_predictor_index

In [4]:
def reader(path_params):
    with open(path_params, "r") as file:
        params = json.load(file)
    print(params)
    return params

In [5]:
def save_list_with_pickle(my_list, file_name):
    with open(file_name, "wb") as file:
        pickle.dump(my_list, file)

In [6]:
def generate_data_set(sample_size, feature_number, label_number, avg_label_number):
    inputs_, outputs_ = make_multilabel_classification(
        n_samples=sample_size + 1,
        n_features=feature_number,
        n_classes=label_number,
        n_labels=avg_label_number,
        allow_unlabeled=False,
        return_indicator=True,
    )
    return inputs_, outputs_

In [7]:
params_path = "params/experiment_2_d.json"
result_path = "results/"
result_name = "experiment_2_d.pkl"

In [8]:
params_global = reader(params_path)

{'data': {'sample_sizes': [1000], 'feature_number': 100, 'label_number': 10, 'avg_label_number': 5, 'cal_size': 0.3, 'proper_cal_size': 0.5}, 'predictor': {'n_estimators_min': 10, 'n_estimators_max': 100, 'criterion': ['gini', 'entropy'], 'lam_numbers': [10, 20, 30, 40, 50]}, 'risk': {'control_level': 0.2}}


In [9]:
complete_n_estimators_grid = np.arange(
    params_global["predictor"]["n_estimators_min"],
    params_global["predictor"]["n_estimators_max"] + 1,
)
np.random.shuffle(complete_n_estimators_grid)
n_estimators_grids = [
    complete_n_estimators_grid[: int(lam_number / 2)]
    for lam_number in params_global["predictor"]["lam_numbers"]
]
print(n_estimators_grids)

[array([100,  41,  90,  92,  44]), array([100,  41,  90,  92,  44,  76,  19,  60,  96,  15]), array([100,  41,  90,  92,  44,  76,  19,  60,  96,  15,  77,  27,  26,
        39,  73]), array([100,  41,  90,  92,  44,  76,  19,  60,  96,  15,  77,  27,  26,
        39,  73,  56,  74,  91,  36,  86]), array([100,  41,  90,  92,  44,  76,  19,  60,  96,  15,  77,  27,  26,
        39,  73,  56,  74,  91,  36,  86,  13,  85,  78,  38,  14])]


In [10]:
save_list_with_pickle(n_estimators_grids, "results/parameter_grid_2.pkl")

In [11]:
parameter_grids = [
    np.array(
        [
            {"n_estimators": n_estimators, "criterion": criterion}
            for n_estimators in n_estimators_grid
            for criterion in params_global["predictor"]["criterion"]
        ],
        dtype=object,
    )
    for n_estimators_grid in n_estimators_grids
]
print(parameter_grids)

[array([{'n_estimators': np.int64(100), 'criterion': 'gini'},
       {'n_estimators': np.int64(100), 'criterion': 'entropy'},
       {'n_estimators': np.int64(41), 'criterion': 'gini'},
       {'n_estimators': np.int64(41), 'criterion': 'entropy'},
       {'n_estimators': np.int64(90), 'criterion': 'gini'},
       {'n_estimators': np.int64(90), 'criterion': 'entropy'},
       {'n_estimators': np.int64(92), 'criterion': 'gini'},
       {'n_estimators': np.int64(92), 'criterion': 'entropy'},
       {'n_estimators': np.int64(44), 'criterion': 'gini'},
       {'n_estimators': np.int64(44), 'criterion': 'entropy'}],
      dtype=object), array([{'n_estimators': np.int64(100), 'criterion': 'gini'},
       {'n_estimators': np.int64(100), 'criterion': 'entropy'},
       {'n_estimators': np.int64(41), 'criterion': 'gini'},
       {'n_estimators': np.int64(41), 'criterion': 'entropy'},
       {'n_estimators': np.int64(90), 'criterion': 'gini'},
       {'n_estimators': np.int64(90), 'criterion': '

In [12]:
predictors_ = [
    [OneVsRestClassifier(BasePredictor(**parameter)) for parameter in parameter_grid]
    for parameter_grid in parameter_grids
]

In [13]:
rep_number = 100

In [14]:
results__ = [
    [
        {
            "avg_order_gap": {"Lower": [], "Upper": []},
            "max_order_gap": {"Lower": [], "Upper": []},
            "avg_size_gap": {"Lower": [], "Upper": []},
            "max_size_gap": {"Lower": [], "Upper": []},
            "cardinal": [],
            "size": {
                "Oracle": [],
                "Upper": [],
                "Split": [],
            },
            "FNR": {
                "Oracle": [],
                "Upper": [],
                "Split": [],
            },
        }
        for _ in predictors_
    ]
    for _ in params_global["data"]["sample_sizes"]
]

In [15]:
results__

[[{'avg_order_gap': {'Lower': [], 'Upper': []},
   'max_order_gap': {'Lower': [], 'Upper': []},
   'avg_size_gap': {'Lower': [], 'Upper': []},
   'max_size_gap': {'Lower': [], 'Upper': []},
   'cardinal': [],
   'size': {'Oracle': [], 'Upper': [], 'Split': []},
   'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
  {'avg_order_gap': {'Lower': [], 'Upper': []},
   'max_order_gap': {'Lower': [], 'Upper': []},
   'avg_size_gap': {'Lower': [], 'Upper': []},
   'max_size_gap': {'Lower': [], 'Upper': []},
   'cardinal': [],
   'size': {'Oracle': [], 'Upper': [], 'Split': []},
   'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
  {'avg_order_gap': {'Lower': [], 'Upper': []},
   'max_order_gap': {'Lower': [], 'Upper': []},
   'avg_size_gap': {'Lower': [], 'Upper': []},
   'max_size_gap': {'Lower': [], 'Upper': []},
   'cardinal': [],
   'size': {'Oracle': [], 'Upper': [], 'Split': []},
   'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
  {'avg_order_gap': {'Lower': [], 'Upper': []},
   'max

In [16]:
for sample_size, results_ in zip(params_global["data"]["sample_sizes"], results__):
    for rep_index in tqdm(range(rep_number)):
        inputs_, outputs_ = generate_data_set(
            sample_size,
            params_global["data"]["feature_number"],
            params_global["data"]["label_number"],
            params_global["data"]["avg_label_number"],
        )

        (
            (scaled_inputs_train, outputs_train),
            (scaled_inputs_calibration, outputs_calibration),
            (
                scaled_inputs_selection,
                scaled_inputs_proper_cal,
                outputs_selection,
                outputs_proper_cal,
            ),
            (scaled_input_test, output_test),
        ) = resample_data_set(
            inputs_,
            outputs_,
            params_global["data"]["cal_size"],
            params_global["data"]["proper_cal_size"],
        )
        for predictors, results in zip(predictors_, results_):
            for predictor in predictors:
                predictor.fit(scaled_inputs_train, outputs_train)

            probas_calibration_per_predictor = [
                predictor.predict_proba(scaled_inputs_calibration)
                for predictor in predictors
            ]
            proba_test_per_predictor = [
                predictor.predict_proba(scaled_input_test) for predictor in predictors
            ]
            oracle_prediction_set = compute_oracle_prediction_set(
                outputs_calibration,
                probas_calibration_per_predictor,
                output_test,
                proba_test_per_predictor,
                params_global["risk"]["control_level"],
            )
            upper_prediction_set = compute_upper_prediction_set(
                outputs_calibration,
                probas_calibration_per_predictor,
                proba_test_per_predictor,
                params_global["risk"]["control_level"],
            )

            probas_selection_per_predictor = [
                predictor.predict_proba(scaled_inputs_selection)
                for predictor in predictors
            ]
            split_predictor_index = select_predictor_index(
                outputs_selection,
                probas_selection_per_predictor,
                params_global["risk"]["control_level"],
            )
            split_prediction_set = compute_prediction_set(
                predictors[split_predictor_index],
                scaled_inputs_proper_cal,
                outputs_proper_cal,
                scaled_input_test,
                params_global["risk"]["control_level"],
            )

            oracle_fnr = (
                np.logical_and(output_test, np.logical_not(oracle_prediction_set)).sum(
                    axis=1
                )
                / output_test.sum(axis=1)
            ).item()
            upper_fnr = (
                np.logical_and(output_test, np.logical_not(upper_prediction_set)).sum(
                    axis=1
                )
                / output_test.sum(axis=1)
            ).item()
            split_fnr = (
                np.logical_and(output_test, np.logical_not(split_prediction_set)).sum(
                    axis=1
                )
                / output_test.sum(axis=1)
            ).item()

            results["FNR"]["Oracle"].append(oracle_fnr)
            results["FNR"]["Upper"].append(upper_fnr)
            results["FNR"]["Split"].append(split_fnr)

            results["size"]["Oracle"].append(oracle_prediction_set.sum())
            results["size"]["Upper"].append(upper_prediction_set.sum())
            results["size"]["Split"].append(split_prediction_set.sum())

            lower_orders, lower_sizes = compute_lower_orders_and_sizes(
                outputs_calibration,
                probas_calibration_per_predictor,
                proba_test_per_predictor,
                params_global["risk"]["control_level"],
            )
            oracle_orders, oracle_sizes = compute_oracle_orders_and_sizes(
                outputs_calibration,
                probas_calibration_per_predictor,
                output_test,
                proba_test_per_predictor,
                params_global["risk"]["control_level"],
            )
            upper_orders, upper_sizes = compute_upper_orders_and_sizes(
                outputs_calibration,
                probas_calibration_per_predictor,
                proba_test_per_predictor,
                params_global["risk"]["control_level"],
            )

            results["avg_order_gap"]["Lower"].append(
                (oracle_orders - lower_orders).mean()
            )
            results["avg_order_gap"]["Upper"].append(
                (upper_orders - oracle_orders).mean()
            )
            results["max_order_gap"]["Lower"].append(
                (oracle_orders - lower_orders).max()
            )
            results["max_order_gap"]["Upper"].append(
                (upper_orders - oracle_orders).max()
            )
            results["avg_size_gap"]["Lower"].append((oracle_sizes - lower_sizes).mean())
            results["avg_size_gap"]["Upper"].append((upper_sizes - oracle_sizes).mean())
            results["max_size_gap"]["Lower"].append((oracle_sizes - lower_sizes).max())
            results["max_size_gap"]["Upper"].append((upper_sizes - oracle_sizes).max())

            min_upper_size = np.min(upper_sizes)
            predictor_number = len(probas_calibration_per_predictor)
            predictor_index_set = np.arange(predictor_number)[
                lower_sizes <= min_upper_size
            ]
            cardinal = len(predictor_index_set)
            results["cardinal"].append(cardinal)

100%|██████████| 100/100 [6:33:45<00:00, 236.26s/it] 


In [17]:
save_list_with_pickle(results__, result_path + result_name)